# Train Silicat v3 on Colab

Pipeline: build tokenized data -> pretrain (resumable) -> chat fine-tune -> download.
Select a GPU runtime and `Runtime -> Run all`. bf16 autocast is picked automatically (`--precision auto`).
On a CPU-only machine use the same commands (AMX/avx512_bf16 CPUs also get bf16); expect ~8 s/step for the full model.

Checkpoints are written atomically; rerun the pretrain cell to continue (it passes `--resume`).
Chat output is `checkpoints/chat_v3.pt` -- the pretrain checkpoint `latest_v3.pt` is never overwritten.

In [ ]:
BRANCH = 'claude/recent-conversations-visibility-YdFCm'
!git clone --depth 1 -b $BRANCH https://github.com/Leo-ailover/Silicat.git
%cd Silicat
!pip install -q torch numpy tokenizers tqdm
# if the branch carries checkpoint parts, they are reassembled automatically on --resume


Build the 32k tokenizer data (block-level random train/val split; needs `data/corpus_v2.txt`).

In [ ]:
!python -m silicat.dataset --v2

Pretrain the 100.7M v3 model (the `--v3` defaults are 12 layers / 12 heads / 4 KV heads / 768 wide / block 512 / batch 8). `--resume` is safe on a fresh machine.

In [ ]:
!python -m silicat.train --stage pretrain --v3 --lr 3e-4 --warmup 400 --max-steps 10000 \
    --eval-interval 500 --save-interval 200 --wsd --resume

Chat fine-tune on `data/silicat_chat_v3.jsonl` (default for `--v3`); picks the checkpoint with the best held-out loss.

In [ ]:
!python -m silicat.train --stage chat --v3 --batch-size 8 --chat-epochs 3

Download the chat model (fp16 export, tied weights shared) and the tokenizer.

In [ ]:
!python -m silicat.halve --src checkpoints/chat_v3.pt --dst checkpoints/chat_v3.fp16.pt
!cd checkpoints && tar czf tokenizer_v2.tgz tokenizer_v2
from google.colab import files
files.download('checkpoints/chat_v3.fp16.pt')
files.download('checkpoints/tokenizer_v2.tgz')